In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from sqlalchemy import create_engine, text, Column, Integer, String, Float, Text
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, Image, HTML
import pandas as pd
import json
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
import requests

load_dotenv()

# Check all api key's
try:
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    ELEVENLABS_API_KEY = os.environ["ELEVENLABS_API_KEY"]
    SPOONACULAR_API_KEY = os.environ["SPOONACULAR_API_KEY"]
    print("Loading API keys done.")
except KeyError as e:
    raise EnvironmentError(f"Missing api key: {e}")

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

Loading API keys done.


In [2]:
# Define the database class  
Base = declarative_base()

# User Profile Table
class UserProfile(Base):
    __tablename__ = 'user_profile'

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    weight = Column(Float)  # in kg
    height = Column(Float)  # in cm
    age = Column(Integer)   # 'Male' / 'Female'
    gender = Column(String)
    activity_level = Column(String)         # e.g., 'sedentary', 'active'
    goal = Column(String)              # e.g., 'weight_loss', 'muscle_gain'
    dietary_restrictions = Column(String)   # e.g., 'gluten, lactose' (comma separated)

# Meal History Table (for future tracking)
class MealLog(Base):
    __tablename__ = 'meal_log'

    id = Column(Integer, primary_key=True)
    date = Column(String, default=lambda: datetime.now().isoformat())
    recipe_name = Column(String)
    calories = Column(Float)
    protein = Column(Float)
    carbs = Column(Float)
    fat = Column(Float)
    user_rating = Column(Integer)   # rating 1-10
    user_notes = Column(Text, nullable=True)

# Create nowaste.db file
engine = create_engine('sqlite:///nowaste.db')
Base.metadata.create_all(engine)

# Create session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()

print("Database ready")

Database ready


In [3]:
# --- Widget Definitions ---
w_name = widgets.Text(description="Name:")
w_weight = widgets.FloatText(description="Weight (kg):", value=70.0)
w_height = widgets.FloatText(description="Height (cm):", value=175.0)
w_age = widgets.IntText(description="Age:", value=30)
w_gender = widgets.Dropdown(options=['Male', 'Female'], description="Gender:", )

# Activity levels mapped to descriptions
w_activity = widgets.Dropdown(
    options=[
        ('Sedentary (Office job, no sports)', 'sedentary'),
        ('Moderate (Exercise 1-3x/week)', 'moderate'),
        ('Active (Exercise 4-5x/week)', 'active'),
        ('Very Active (Athlete)', 'very_active')
    ],
    description="Activity:"
)

# Goals
w_goal = widgets.Dropdown(
    options=[
        ('Weight Loss (Reduction)', 'weight_loss'),
        ('Maintain Weight', 'maintenance'),
        ('Muscle Gain (Bulk)', 'muscle_gain')
    ],
    description="Goal:"
)

w_restrictions = widgets.Textarea(
    description="Intolerances:",
    placeholder="e.g. gluten, peanuts (leave empty if none)",
    
)

btn_save = widgets.Button(description="Save Profile", button_style='success')
output = widgets.Output()

# --- Callback Function to Save Data ---
def save_profile_to_db(b):
    """
    Save user info to database
    """
    with output:
        clear_output()
        
        # Check if a user already exists (assuming single-user app for now)
        existing_user = session.query(UserProfile).first()
        
        if not existing_user:
            existing_user = UserProfile()
            session.add(existing_user)
            print("Creating new profile...")
        else:
            print("Updating existing profile...")
        
        # Assign values from widgets to the database object
        existing_user.name = w_name.value
        existing_user.weight = w_weight.value
        existing_user.height = w_height.value
        existing_user.age = w_age.value
        existing_user.gender = w_gender.value
        existing_user.activity_level = w_activity.value
        existing_user.goal = w_goal.value
        existing_user.dietary_restrictions = w_restrictions.value
        
        # Commit changes to the database
        try:
            session.commit()
            print(f"SUCCESS! Profile for {existing_user.name} saved.")
            print(f"Goal: {existing_user.goal} | Weight: {existing_user.weight}kg")
        except Exception as e:
            session.rollback()
            print(f"Error saving to database: {e}")

btn_save.on_click(save_profile_to_db)

# --- Display the Form ---
display(widgets.VBox([
    widgets.HTML("<h3>User Profile Setup</h3>"),
    w_name, w_weight, w_height, w_age, w_gender,
    w_activity, w_goal, w_restrictions,
    btn_save, output
]))

In [10]:
def show_table(model_class, table_name):
    stmt = session.query(model_class).statement
    df = pd.read_sql(stmt, session.bind) #type: ignore
    
    if not df.empty:
        display(df)
    else:
        print("(Tabela jest pusta)")
    print("\n")

show_table(UserProfile, "User Profile")

show_table(MealLog, "Meal Log")

,id,name,weight,height,age,gender,activity_level,goal,dietary_restrictions
0,1,Adam,70.0,175.0,30,Male,sedentary,weight_loss,




(Tabela jest pusta)




In [5]:
# db tests
def run_system_health_check():
    print("STARTING SYSTEM TESTS...\n")
    
    # --- TEST 1: File Existence ---
    db_file = 'nowaste.db'
    if os.path.exists(db_file):
        print(f"PASS: Database file '{db_file}' exists.")
    else:
        print(f"FAIL: Database file '{db_file}' not found.")
        return # Stop tests if file is missing

    # --- TEST 2: Database Connection ---
    try:
        # Try to execute a simple SQL query
        session.execute(text("SELECT 1"))
        print("PASS: Database connection established.")
    except Exception as e:
        print(f"FAIL: Database connection failed. Error: {e}")
        return

    # --- TEST 3: CRUD Logic (Create, Read, Update, Delete) ---
    print("\nRunning CRUD Test (on temporary data)...")
    try:
        # 3.1 CREATE
        test_user = UserProfile(
            name="TestUnit_Ghost", 
            weight=100.0, 
            goal="test", 
            dietary_restrictions="none"
        )
        session.add(test_user)
        session.commit()
        
        # 3.2 READ
        retrieved_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert retrieved_user is not None, "Failed to retrieve created user"
        assert retrieved_user.weight == 100.0, "Weight mismatch" # type: ignore
        
        # 3.3 UPDATE
        retrieved_user.goal = "updated_goal" # type: ignore
        session.commit()
        updated_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert updated_user.goal == "updated_goal", "Update failed" # type: ignore
        
        # 3.4 DELETE (Cleanup)
        session.delete(updated_user)
        session.commit()
        deleted_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert deleted_user is None, "Delete failed"
        
        print("PASS: CRUD operations working correctly.")
        
    except AssertionError as ae:
        print(f"FAIL: Assertion failed: {ae}")
        session.rollback()
    except Exception as e:
        print(f"FAIL: CRUD Error: {e}")
        session.rollback()

    # --- TEST 4: Verify Your Real Profile ---
    print("\nVerifying Active User Profile...")
    real_user = session.query(UserProfile).first()
    if real_user:
        print(f"PASS: Active user found: '{real_user.name}'")
        print(f"\tDetails: {real_user.weight}kg, Goal: {real_user.goal}")
    else:
        print("WARNING: No active user profile found. (Did you click 'Save Profile' in the previous step?)")

    print("\nTESTS COMPLETED.")

# Run the tests
run_system_health_check()

STARTING SYSTEM TESTS...

PASS: Database file 'nowaste.db' exists.
PASS: Database connection established.

Running CRUD Test (on temporary data)...
PASS: CRUD operations working correctly.

Verifying Active User Profile...
PASS: Active user found: 'adam'
	Details: 70.0kg, Goal: weight_loss

TESTS COMPLETED.


In [6]:
class IdentifiedItem(BaseModel):
    name_en_clean: str = Field(
        description="ONLY the core ingredient keyword in English "
                    "(e.g. 'eggs', 'milk', 'chicken breast'). "
                    "NO packaging, brands, or containers."
    )
    name_en: str = Field(
        description="Ingredient name in English, human-readable."
    )
    name_pl: str = Field(
        description="Ingredient name in Polish (for UI)."
    )
    quantity_estimated: Optional[str] = Field(
        description="Estimated quantity in Polish (e.g. '2 sztuki', 'ok. 200g'). "
                    "Use null if not possible."
    )
    confidence_level: Literal["high", "medium", "low"] = Field(
        description="Confidence level of visual recognition."
    )
    needs_clarification: bool = Field(
        description="True if the ingredient is ambiguous or unclear."
    )
    is_staple: bool = Field(
        description="True if ingredient is a basic kitchen staple "
                    "(spices, oils, sauces, sugar, flour, etc.)."
    )


class FridgeAnalysis(BaseModel):
    identified_items: List[IdentifiedItem]
    clarification_questions: List[str]
    ready_to_search: bool = Field(
        description="True ONLY if no clarification is needed and "
                    "core ingredients are confidently identified."
    )

STAPLES = {
    # spices
    "salt", "pepper", "black pepper", "white pepper",
    "paprika", "smoked paprika", "chili powder",
    "cumin", "curry powder", "turmeric",
    "oregano", "basil", "thyme", "rosemary",
    "bay leaf", "garlic powder", "onion powder",

    # fats
    "oil", "olive oil", "vegetable oil", "rapeseed oil",
    "sunflower oil", "coconut oil",
    "butter", "margarine", "ghee",

    # sweeteners
    "sugar", "brown sugar", "powdered sugar",
    "honey", "maple syrup", "agave syrup", "sweetener",

    # sauces
    "soy sauce", "ketchup", "mustard", "mayonnaise",
    "bbq sauce", "hot sauce", "sriracha",
    "vinegar", "apple cider vinegar", "balsamic vinegar",

    # technical
    "baking powder", "baking soda", "yeast",
    "cornstarch", "gelatin", "flour", "wheat flour"
}


def create_openai_file(file_path: str) -> str:
    """
    Uploads a local image file to OpenAI Files API for vision purposes.
    Returns file_id.
    """
    with open(file_path, "rb") as file_content:
        result = client.files.create(
            file=file_content,
            purpose="vision",
        )
    return result.id


def analyze_fridge_process(file_id: str) -> Optional[FridgeAnalysis]:
    """
    Initial fridge analysis using GPT vision.
    Returns FridgeAnalysis or None on error.
    """

    system_prompt = """
    Jesteś inteligentnym asystentem kulinarnym analizującym zdjęcie lodówki lub blatu.

    TWOJE ZADANIE:
    1. Zidentyfikuj wszystkie widoczne produkty spożywcze.
    2. Każdy produkt opisz jako osobny obiekt.

    ZASADY NAZW:
    - Pole `name_en_clean` MUSI zawierać WYŁĄCZNIE główny składnik kulinarny po angielsku
    (np. 'eggs', 'milk', 'chicken breast', 'bell pepper').
    - NIGDY nie dodawaj informacji o opakowaniu, marce ani formie
    ('carton', 'jar', 'package').
    - `name_en` może być bardziej opisowe, ale zgodne z tym samym składnikiem.
    - `name_pl` ma być przyjazne dla użytkownika.

    STAPLES (POMIJANE PRZY WYSZUKIWANIU PRZEPISÓW):
    Oznacz `is_staple = true` dla produktów, które:
    - są przyprawami (sól, pieprz, papryka, oregano, curry)
    - są olejami lub tłuszczami (olej, oliwa, masło)
    - są sosami (ketchup, musztarda, majonez, sos sojowy)
    - są słodzikami (cukier, miód, syrop klonowy)
    - są produktami technicznymi (mąka, drożdże, proszek do pieczenia)

    ILOŚĆ:
    - Podaj realistyczne oszacowanie ilości po polsku.
    - Jeśli to niemożliwe, użyj null.

    PEWNOŚĆ I PYTANIA:
    - Jeśli nie masz pewności, ustaw confidence_level na 'medium' lub 'low'.
    - Jeśli produkt jest niejednoznaczny (np. zawartość słoika),
    ustaw needs_clarification = true i dodaj pytanie po polsku.

    WARUNEK GOTOWOŚCI:
    - ready_to_search = true TYLKO jeśli:
    • brak otwartych pytań
    • kluczowe składniki mają confidence 'high' lub 'medium'
    """

    user_prompt = "Zidentyfikuj produkty spożywcze widoczne na zdjęciu."

    msg = [
        {"role": "developer", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": user_prompt},
                {"type": "input_image", "file_id": file_id},
            ],
        }
    ]

    try:
        response = client.responses.parse(
            model="gpt-5-mini",
            input=msg,
            text_format=FridgeAnalysis
        )

        return response.output_parsed

    except Exception as e:
        print(f"[ERROR] Fridge analysis failed: {e}")
        return None


def refine_analysis_process(
    file_id: str,
    previous_analysis: FridgeAnalysis,
    user_answers: str
) -> Optional[FridgeAnalysis]:
    """
    Refinement loop: updates analysis based on user answers.
    """

    system_prompt = """
    Jesteś inteligentnym asystentem kulinarnym.

    Użytkownik odpowiedział na Twoje pytania dotyczące poprzedniej analizy zdjęcia.

    ZASADY:
    - Zachowaj wszystkie poprawnie zidentyfikowane składniki.
    - Popraw lub uzupełnij TYLKO elementy, które były niejasne.
    - Nie dodawaj nowych produktów, jeśli nie wynikają z obrazu lub odpowiedzi użytkownika.

    NAZWY:
    - `name_en_clean` MUSI pozostać czystą nazwą składnika (bez opakowań).
    - Staples oznacz jako `is_staple = true`.

    CEL:
    - Zwróć kompletny obiekt typu FridgeAnalysis.
    - Jeśli wszystkie kluczowe składniki są już jasne,
    ustaw ready_to_search = true.
    """

    user_content = (
        f"Poprzednia analiza (JSON):\n{previous_analysis.model_dump_json()}\n\n"
        f"Odpowiedzi użytkownika:\n{user_answers}\n\n"
        "Zaktualizuj analizę."
    )

    msg = [
        {"role": "developer", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": user_content},
                {"type": "input_image", "file_id": file_id},
            ],
        }
    ]

    try:
        response = client.responses.parse(
            model="gpt-5-mini",
            input=msg,
            text_format=FridgeAnalysis
        )

        return response.output_parsed

    except Exception as e:
        print(f"[ERROR] Refinement failed: {e}")
        return None


In [ ]:
# --- Zmienne globalne do przechowywania stanu analizy ---
current_file_id = None
current_analysis_result = None

# --- Widgety UI ---
btn_upload = widgets.FileUpload(accept='image/*', multiple=False, description='Wgraj Zdjęcie')
output_vision = widgets.Output()

# Sekcja Chatu (domyślnie ukryta)
lbl_chat = widgets.HTML("<b>AI ma pytania:</b>")
txt_answer = widgets.Textarea(placeholder='Odpisz tutaj...', layout=widgets.Layout(width='100%', height='60px'))
btn_send_answer = widgets.Button(description='Wyślij odpowiedź', button_style='info', icon='paper-plane')
btn_skip = widgets.Button(description='Pomiń (Gotuj z tego co pewne)', button_style='warning')
box_chat = widgets.VBox([lbl_chat, txt_answer, widgets.HBox([btn_send_answer, btn_skip])])
box_chat.layout.display = 'none'

def display_fridge_result(result: FridgeAnalysis):
    """Renderuje wyniki analizy w ładny sposób."""
    if not result: return

    # Rozdzielamy składniki na główne i bazowe (staples)
    main_items = [i for i in result.identified_items if not i.is_staple and not i.needs_clarification]
    staple_items = [i for i in result.identified_items if i.is_staple]
    unclear_items = [i for i in result.identified_items if i.needs_clarification]

    html = "<h3>Wynik Analizy</h3>"
    
    # 1. Główne składniki
    if main_items:
        html += "<h4>Główne Produkty (Do bazy przepisu):</h4><ul>"
        for item in main_items:
            qty = f" ({item.quantity_estimated})" if item.quantity_estimated else ""
            html += f"<li><b>{item.name_pl}</b> {qty} <span style='color:gray; font-size:0.8em'>[{item.name_en_clean}]</span></li>"
        html += "</ul>"
    
    # 2. Staples (Przyprawy, oleje itp.)
    if staple_items:
        html += "<h4>Spiżarnia (Przyprawy/Dodatki):</h4><div style='color:#555; font-size:0.9em;'>"
        html += ", ".join([item.name_pl for item in staple_items])
        html += "</div>"

    # 3. Status
    if result.ready_to_search:
        html += "<div style='margin-top:10px; padding:10px; background:#e8f5e9; color:green; border-radius:5px;'><b>STATUS: GOTOWY DO SZUKANIA PRZEPISÓW</b></div>"
        box_chat.layout.display = 'none'
    else:
        html += "<div style='margin-top:10px; padding:10px; background:#fff3e0; color:#e65100; border-radius:5px;'><b>STATUS: WYMAGANE DOPYTANIE</b></div>"
        
        # Wyświetl pytania
        q_list = "".join([f"<li>{q}</li>" for q in result.clarification_questions])
        lbl_chat.value = f"<b>Pytania od AI:</b><ul>{q_list}</ul>"
        box_chat.layout.display = 'block'

    display(HTML(html))

def on_upload(change):
    """Obsługa wgrania pliku"""
    global current_file_id, current_analysis_result
    output_vision.clear_output()
    box_chat.layout.display = 'none'
    txt_answer.value = ""

    if not btn_upload.value: return

    with output_vision:
        # 1. Pobierz plik
        upl_file = btn_upload.value[0] if isinstance(btn_upload.value, tuple) else list(btn_upload.value.values())[0]
        content = upl_file['content']
        
        # 2. Pokaż zdjęcie
        display(Image(data=content, width=300))
        print("Analizuję zdjęcie (gpt-5-mini)...")
        
        # 3. Zapisz temp i wyślij
        temp_name = "temp_fridge.jpg"
        with open(temp_name, "wb") as f: f.write(content)
        
        try:
            current_file_id = create_openai_file(temp_name)
            current_analysis_result = analyze_fridge_process(current_file_id)
            
            clear_output(wait=True)
            display(Image(data=content, width=300))
            display_fridge_result(current_analysis_result)
            
        except Exception as e:
            print(f"Błąd: {e}")
        finally:
            if os.path.exists(temp_name): os.remove(temp_name)

def on_reply(b):
    """Obsługa odpowiedzi na pytania"""
    global current_analysis_result
    if not current_file_id or not txt_answer.value: return
    
    with output_vision:
        print("Aktualizuję analizę...")
        new_result = refine_analysis_process(current_file_id, current_analysis_result, txt_answer.value)
        if new_result:
            current_analysis_result = new_result
            clear_output(wait=True)
            # Ponowne wyświetlenie zdjęcia (żeby nie zniknęło)
            display_fridge_result(new_result)

def on_skip(b):
    """Wymuszenie gotowości (ignorowanie pytań)"""
    global current_analysis_result
    if current_analysis_result:
        current_analysis_result.ready_to_search = True
        with output_vision:
            clear_output(wait=True)
            display_fridge_result(current_analysis_result)

# Podpięcie zdarzeń
btn_upload.observe(on_upload, names='value')
btn_send_answer.on_click(on_reply)
btn_skip.on_click(on_skip)

display(widgets.VBox([
    widgets.HTML("<h3>Krok 2: Analiza Lodówki</h3>"),
    btn_upload,
    output_vision,
    box_chat
]))

In [ ]:
import json
import requests
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pydantic import BaseModel
from typing import List

# --- 1. Modele Danych ---

class RecipeSearchPlan(BaseModel):
    ingredients: list[str]
    reason: str 

class SearchPlans(BaseModel):
    plans: list[RecipeSearchPlan]

# --- 2. Logika Planowania (AI) ---

def plan_recipe_search(ingredients: list[str], user_profile=None) -> SearchPlans:
    """AI tworzy 3 zestawy składników."""
    profile_context = ""
    if user_profile:
        profile_context = f"User Goal: {user_profile.goal}, Restrictions: {user_profile.dietary_restrictions}"

    system_prompt = f"""
    You are a culinary expert.
    From the provided ingredients, create 3 reasonable ingredient sets
    (each max 4 ingredients) that could realistically form a tasty dish.

    Rules:
    - Ignore spices, oils, sauces etc. (assume user has them).
    - Prefer protein + carb + vegetable combinations.
    - Do NOT try to use all ingredients at once.
    - For each set try to use diffrent ingredients if it is possible.
    - Try to create sets for realistic dishes.
    - Provide a short 'reason' in Polish describing the dish idea (e.g. 'Jajecznica z warzywami').
    {profile_context}
    """

    user_prompt = {"available_ingredients": ingredients}

    try:
        response = client.responses.parse(
            model="gpt-5-mini",
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_prompt)}
            ],
            text_format=SearchPlans
        )
        return response.output_parsed
    except Exception as e:
        print(f"[ERROR] AI Planning: {e}")
        return None

# --- 3. Logika API (Spoonacular) ---

# def fetch_spoonacular_recipes(plans: SearchPlans, user_profile):
#     """
#     Zwraca listę krotek: [(plan_obiekt, lista_przepisow), ...]
#     """
#     results_list = []
    
#     base_params = {
#         "apiKey": SPOONACULAR_API_KEY,
#         "number": 2,
#         "fillIngredients": True,
#         "ignorePantry": True,
#         "addRecipeInformation": True,
#         "addRecipeNutrition": True,
#         "instructionsRequired": True,
#         "ranking": 2
#     }
    
#     if user_profile:
#         if user_profile.goal == 'muscle_gain':
#             base_params['minProtein'] = 25
#             base_params['sort'] = 'protein'
#         elif user_profile.goal == 'weight_loss':
#             base_params['maxCalories'] = 900
#         if user_profile.dietary_restrictions:
#             base_params['intolerances'] = user_profile.dietary_restrictions

#     for i, plan in enumerate(plans.plans):
#         # 1. BEZPIECZNIK: Przycinamy do 4 składników
#         safe_ingredients = plan.ingredients[:4]
#         plan.ingredients = safe_ingredients 
        
#         query_str = ",".join(safe_ingredients)
#         params = base_params.copy()
#         params['includeIngredients'] = query_str

#         # print("Params for API:")
#         # print(params)
        
#         try:
#             r = requests.get("https://api.spoonacular.com/recipes/findByIngredients", params=params)
#             r.raise_for_status()
#             data = r.json()

#             # print(f"📦 [DEBUG] JSON Response:")
#             # print(json.dumps(data, indent=2, ensure_ascii=False)) # Pretty print JSON
#             # print("-" * 50)
            
#             if data['results']:
#                 results_list.append((plan, data['results']))
#             else:
#                 # Opcjonalnie: logowanie pustych wyników w konsoli
#                 # print(f"   -> Brak wyników dla: {safe_ingredients}")
#                 pass
                
#         except Exception as e:
#             print(f"   -> Błąd API: {e}")
            
#     return results_list


def get_nutrient(rec, name) -> float:
    for n in rec.get("nutrition", {}).get("nutrients", []):
        if n.get("name") == name:
            return n.get("amount", 0)
    return 0

def fetch_spoonacular_recipes(plans: SearchPlans, user_profile):
    """
    Zwraca listę krotek: [(plan_obiekt, lista_przepisow), ...]
    Używa podejścia: findByIngredients -> informationBulk -> filtrowanie lokalne.
    """
    results_list = []
    
    # KROK 1: Parametry wyszukiwania po składnikach
    find_params = {
        "apiKey": SPOONACULAR_API_KEY,
        "number": 5,      # Pobieramy więcej do filtracji
        "ranking": 2,     # Minimalizuj brakujące składniki
        "ignorePantry": True
    }

    for i, plan in enumerate(plans.plans):
        safe_ingredients = plan.ingredients[:4]
        plan.ingredients = safe_ingredients 
        
        # Budowanie zapytania
        current_find_params = find_params.copy()
        current_find_params['ingredients'] = ",".join(safe_ingredients)
        
        try:
            # 1. Szukamy pomysłów (findByIngredients)
            r_find = requests.get("https://api.spoonacular.com/recipes/findByIngredients", params=current_find_params)
            r_find.raise_for_status()
            find_data = r_find.json()
            
            if not find_data:
                continue

            # 2. Pobieramy szczegóły dla znalezionych ID (informationBulk)
            recipe_ids = [str(item['id']) for item in find_data]
            ids_string = ",".join(recipe_ids)
            
            bulk_params = {
                "apiKey": SPOONACULAR_API_KEY,
                "ids": ids_string,
                "includeNutrition": True
            }
            
            r_bulk = requests.get("https://api.spoonacular.com/recipes/informationBulk", params=bulk_params)
            r_bulk.raise_for_status()
            bulk_data = r_bulk.json()
            
            # 3. Filtrowanie przy użyciu get_nutrient
            valid_recipes = []
            
            for recipe in bulk_data:
                # Sprawdzenie instrukcji
                if not recipe.get('analyzedInstructions') and not recipe.get('instructions'):
                    continue

                # Wyciągamy kluczowe makro do filtracji
                cal = get_nutrient(recipe, "Calories")
                prot = get_nutrient(recipe, "Protein")
                
                # Logika User Profile
                if user_profile:
                    # Intolerancje
                    if user_profile.dietary_restrictions:
                        intolerances = [x.lower() for x in user_profile.dietary_restrictions.split(',')]
                        if 'gluten' in intolerances and not recipe.get('glutenFree', False):
                            continue

                    # Cele
                    if user_profile.goal == 'muscle_gain':
                        if prot < 25: continue # Za mało białka
                    elif user_profile.goal == 'weight_loss':
                        if cal > 900: continue # Za dużo kalorii

                valid_recipes.append(recipe)
            
            if valid_recipes:
                results_list.append((plan, valid_recipes[:2])) # Bierzemy max 2 najlepsze

        except Exception as e:
            print(f"Błąd API dla {safe_ingredients}: {e}")
            
    return results_list

# --- 4. Interfejs (UI) ---

btn_generate = widgets.Button(
    description='🚀 Generuj Przepisy', 
    button_style='success', 
    layout=widgets.Layout(width='300px')
)
output_final = widgets.Output()



def on_click_generate(b):
    output_final.clear_output()
    
    if not current_analysis_result or not current_analysis_result.ready_to_search:
        with output_final: print("⚠️ Najpierw wgraj zdjęcie (Krok 2).")
        return
    
    ingredients_input = [
        item.name_en_clean for item in current_analysis_result.identified_items
        if not item.is_staple and not item.needs_clarification
    ]
    
    if not ingredients_input:
        with output_final: print("⚠️ Brak głównych składników.")
        return

    user = session.query(UserProfile).first()

    with output_final:
        print("🧠 AI planuje posiłki...")
        plans_object = plan_recipe_search(ingredients_input, user)
        
        if not plans_object: return
        
        # --- DEBUG: WIZUALIZACJA PLANÓW AI ---
        debug_html = "<div style='background:#f3f3f3; padding:10px; border-left:4px solid #f0ad4e; margin-bottom:20px;'>"
        debug_html += "<h4 style='margin-top:0; color:#d35400;'>🛠️ [DEBUG] AI wygenerowało 3 strategie:</h4>"
        for i, plan in enumerate(plans_object.plans):
            ing_str = ", ".join([f"<b>{ing}</b>" for ing in plan.ingredients])
            debug_html += f"<div style='margin-bottom:5px;'>Strategia {i+1}: {ing_str} <i style='color:#777'>({plan.reason})</i></div>"
        debug_html += "</div>"
        display(HTML(debug_html))
        # -------------------------------------

        print("🌍 Pobieranie przepisów ze Spoonacular...")
        final_results = fetch_spoonacular_recipes(plans_object, user)
        
        # Nie czyścimy outputu całkowicie, żeby zostawić DEBUG widoczny
        # clear_output(wait=True) 
        
        if not final_results:
            print("❌ Nie znaleziono przepisów dla żadnego z planów.")
            return

        # RENDEROWANIE WYNIKÓW
        html_content = ""
        for plan, recipes in final_results:
            
            ing_badges = "".join([
                f"<span style='background:#e0f7fa; color:#006064; padding:2px 6px; border-radius:4px; margin-right:5px; font-size:0.8em;'>{ing}</span>"
                for ing in plan.ingredients
            ])

            html_content += f"""
            <div style='margin-top:25px; border-top:1px solid #ddd; padding-top:15px;'>
                <div style='margin-bottom:15px;'>🛒 Baza: {ing_badges}</div>
                <div style='display:flex; flex-wrap:wrap; gap:15px;'>
            """
            
            for rec in recipes:
                cal = float(get_nutrient(rec, "Calories"))
                prot = float(get_nutrient(rec, "Protein"))
                fat = float(get_nutrient(rec, "Fat"))
                carbs = float(get_nutrient(rec, "Carbohydrates"))

                html_content += f"""
                <div style="width:240px; border:1px solid #ccc; padding:10px; border-radius:8px; background:white; box-shadow: 2px 2px 5px rgba(0,0,0,0.05);">
                    <img src="{rec['image']}" style="width:100%; height:130px; object-fit:cover; border-radius:5px;">
                    <h5 style="margin:8px 0; white-space: nowrap; overflow: hidden; text-overflow: ellipsis;">{rec['title']}</h5>
                    
                    <div style="font-size:0.85em; color:#555; margin-bottom:8px;">
                        🔥 <b>{cal} kcal</b> | 🥩 <b>{prot}g białka </b>
                    </div>

                    <div style="font-size:0.85em; color:#555; margin-bottom:8px;">
                        🥑 <b>{fat} g</b> | 🍞 <b>{carbs}g białka </b>
                    </div>
                    
                    <a href="{rec['sourceUrl']}" target="_blank" 
                       style="display:block; background:#007bff; color:white; text-align:center; padding:6px; border-radius:4px; text-decoration:none; font-size:0.9em;">
                       Zobacz Przepis
                    </a>
                </div>
                """
            html_content += "</div></div>"
            
        display(HTML(html_content))

btn_generate.on_click(on_click_generate)

display(widgets.VBox([
    widgets.HTML("<h3>Krok 3: Inteligentny Planer + Debug</h3>"),
    btn_generate,
    output_final
]))